# Anchored Sentiment Trajectory Analysis

This notebook computes and visualises an anchored sentiment trajectory over a literary text.

The method represents each text segment with a sentence-transformer model and compares it with two sets of reference examples: positive anchors and negative anchors. Each segment receives a score based on the difference between its average similarity to the positive anchors and its average similarity to the negative anchors.

A positive score means that the segment is closer to the positive anchors. A negative score means that the segment is closer to the negative anchors.

## Workflow

This notebook follows five main steps:

1. Prepare or load a segmented version of the text.
2. Store the segments in a standard dataframe, `segments_df`.
3. Load a sentence-transformer model.
4. Compute anchored sentiment scores for each segment.
5. Visualise the resulting sentiment trajectory.

## Data and copyright note

The full literary text and the anchor sentences are not included in this repository because they contain copyrighted material. To run the notebook locally, provide:

- a prepared local text file or structured source file;
- a local anchor CSV file containing positive and negative anchor sentences;
- a local sentence-transformer model path, or a model name available through `sentence-transformers`.

The notebook is designed so that the preprocessing step can be adapted to different source formats, while the scoring and plotting steps operate on the standard `segments_df` format.

In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path
from typing import List, Tuple
from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup

## Preprocessing and segmentation

The sentiment-scoring section expects a dataframe named `segments_df` with at least the following columns:

- `segment_text`: the text segment to be scored;
- `chapter`: the chapter or section label aligned with the segment;
- `segment_type`: optional information about the segment type, such as prose, dialogue, verse, or unknown.

Different source formats require different preprocessing rules. Where possible, structured formats such as TEI/XML or HTML should be preferred because paragraph, chapter, and verse boundaries may already be marked. When only plain text is available, source-specific rules must be checked and documented.

The optional parser cells below illustrate possible preprocessing routes. The default workflow in this notebook uses a prepared plain-text file with explicit chapter markers.

The prepared plain-text parser below is used for the local LOTR file in this project. It assumes chapter markers in the form `###CHAPTER:` and a local paragraph-start convention based on leading spaces. These rules are source-specific.

### TEI/XML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

TEI_PATH = Path("data/source.xml")

def segments_from_tei(path: Path) -> pd.DataFrame:
    """
    Extract prose paragraphs and verse blocks from a TEI/XML file.

    This is a template parser. TEI structures vary, so tag names and
    attributes may need to be adapted for a specific corpus.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "xml")

    rows = []

    for div in soup.find_all("div"):
        chapter_head = div.find("head")
        chapter = chapter_head.get_text(" ", strip=True) if chapter_head else "Unknown"

        for element in div.find_all(["p", "lg"], recursive=True):
            if element.name == "p":
                segment_type = "prose"
                text = element.get_text(" ", strip=True)

            elif element.name == "lg":
                segment_type = "verse"
                lines = [line.get_text(" ", strip=True) for line in element.find_all("l")]
                text = " / ".join(lines) if lines else element.get_text(" ", strip=True)

            else:
                continue

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": chapter,
                        "segment_type": segment_type,
                    }
                )

    return pd.DataFrame(rows)

### HTML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

HTML_PATH = Path("data/source.html")

def segments_from_html(path: Path) -> pd.DataFrame:
    """
    Extract paragraphs from an HTML file.

    This works best for HTML/EPUB-derived texts where paragraphs are
    marked with <p> tags. Chapter detection may need to be adapted
    depending on the source.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    rows = []
    current_chapter = "Unknown"

    for element in soup.find_all(["h1", "h2", "h3", "p"]):
        if element.name in ["h1", "h2", "h3"]:
            current_chapter = element.get_text(" ", strip=True)

        elif element.name == "p":
            text = element.get_text(" ", strip=True)

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": current_chapter,
                        "segment_type": "prose",
                    }
                )

    return pd.DataFrame(rows)

## Prepared plain-text parser used in this notebook

In [ ]:
# Cell — Segment corpus

# -------------------------------
# Segmentation parameters
# -------------------------------
MIN_TOK_NARR, MAX_TOK_NARR = 80, 300
MIN_TOK_DIAL, MAX_TOK_DIAL = 60, 180
MAX_DIALOGUE_TURNS = 6

DIALOG_START_CHARS = ('"', "'", "“", "”", "‘", "’", "—", "–", "-", "―")

CORPUS_PATH = Path("data/LotR.txt")

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CORPUS_PATH}. "
        "The source text is not included in the public repository because it contains copyrighted text."
    )


def is_dialogue_first_line(line: str) -> bool:
    """Return True if a line appears to begin with dialogue punctuation."""
    return line.lstrip().startswith(DIALOG_START_CHARS)


def is_paragraph_lead(line: str) -> bool:
    """Return True if the line follows the local five-space paragraph convention."""
    return line.startswith("     ")


def tokenize_count(text: str) -> int:
    """Count word tokens, treating hyphenated words as single tokens."""
    return len(re.findall(r"\b\w+(?:-\w+)*\b", text))


def build_raw_paragraphs(lines: List[str]) -> Tuple[List[str], List[str], List[bool]]:
    """Build raw paragraphs with aligned chapter labels and dialogue flags."""
    paragraphs: List[str] = []
    para_chapters: List[str] = []
    para_is_dialogue: List[bool] = []

    current_chapter = "Unknown"
    buf_lines: List[str] = []
    buf_is_dialogue = False

    def flush_paragraph():
        nonlocal buf_lines, buf_is_dialogue

        if buf_lines:
            paragraphs.append(" ".join(ln.strip() for ln in buf_lines).strip())
            para_chapters.append(current_chapter)
            para_is_dialogue.append(buf_is_dialogue)

        buf_lines = []
        buf_is_dialogue = False

    for raw in lines:
        line = raw.rstrip("\r")

        if line.startswith("###CHAPTER:"):
            flush_paragraph()
            current_chapter = line.replace("###CHAPTER:", "").strip()
            continue

        if line.strip() == "":
            flush_paragraph()
            continue

        if is_paragraph_lead(line):
            flush_paragraph()
            buf_lines = [line]
            buf_is_dialogue = is_dialogue_first_line(line)
        else:
            if buf_lines:
                buf_lines.append(line)
            else:
                buf_lines = [line]
                buf_is_dialogue = is_dialogue_first_line(line)

    flush_paragraph()
    return paragraphs, para_chapters, para_is_dialogue


def merge_dialogue_aware(
    paragraphs: List[str],
    para_chapters: List[str],
    para_is_dialogue: List[bool],
) -> Tuple[List[str], List[str]]:
    """
    Merge raw paragraphs into dialogue-aware analysis segments.

    Chapter boundaries are preserved: segments are never merged across chapters.
    """
    processed_paragraphs: List[str] = []
    chapter_tags: List[str] = []

    seg_buf: List[str] = []
    seg_tokens = 0
    seg_is_dialogue = None
    seg_dialogue_turns = 0
    seg_chapter = None

    def seg_flush():
        nonlocal seg_buf, seg_tokens, seg_is_dialogue, seg_dialogue_turns, seg_chapter

        if seg_buf:
            processed_paragraphs.append(" ".join(seg_buf).strip())
            chapter_tags.append(seg_chapter)

        seg_buf = []
        seg_tokens = 0
        seg_is_dialogue = None
        seg_dialogue_turns = 0
        seg_chapter = None

    for paragraph, chapter, is_dialogue in zip(
        paragraphs,
        para_chapters,
        para_is_dialogue,
    ):
        paragraph_tokens = tokenize_count(paragraph)

        if paragraph_tokens == 0:
            continue

        if seg_buf and chapter != seg_chapter:
            seg_flush()

        if not seg_buf:
            seg_buf = [paragraph]
            seg_tokens = paragraph_tokens
            seg_is_dialogue = is_dialogue
            seg_dialogue_turns = 1 if is_dialogue else 0
            seg_chapter = chapter
            continue

        max_tokens = MAX_TOK_DIAL if seg_is_dialogue else MAX_TOK_NARR
        min_tokens = MIN_TOK_DIAL if seg_is_dialogue else MIN_TOK_NARR

        type_switch = is_dialogue != seg_is_dialogue
        exceeds_limits = (
            seg_tokens + paragraph_tokens > max_tokens
            or (seg_is_dialogue and seg_dialogue_turns >= MAX_DIALOGUE_TURNS)
        )

        if type_switch or exceeds_limits:
            if seg_tokens >= min_tokens:
                seg_flush()

                seg_buf = [paragraph]
                seg_tokens = paragraph_tokens
                seg_is_dialogue = is_dialogue
                seg_dialogue_turns = 1 if is_dialogue else 0
                seg_chapter = chapter
                continue

        seg_buf.append(paragraph)
        seg_tokens += paragraph_tokens

        if is_dialogue and seg_is_dialogue:
            seg_dialogue_turns += 1
        elif type_switch:
            seg_is_dialogue = is_dialogue
            seg_dialogue_turns = 1 if is_dialogue else 0

    seg_flush()
    return processed_paragraphs, chapter_tags


with open(CORPUS_PATH, "r", encoding="utf-8") as file:
    lines = file.read().split("\n")

raw_paragraphs, raw_chapter_tags, raw_is_dialogue = build_raw_paragraphs(lines)

processed_paragraphs, chapter_tags = merge_dialogue_aware(
    raw_paragraphs,
    raw_chapter_tags,
    raw_is_dialogue,
)

print(f"Raw paragraphs: {len(raw_paragraphs)}")
print(f"Processed segments: {len(processed_paragraphs)}")
print(f"Unique chapters: {len(set(chapter_tags))}")

In [ ]:
# Cell — Source-format diagnostics

chapter_marker_count = sum(
    line.startswith("###CHAPTER:")
    for line in lines
)

paragraph_lead_count = sum(
    is_paragraph_lead(line)
    for line in lines
)

blank_line_count = sum(
    line.strip() == ""
    for line in lines
)

source_diagnostics = {
    "chapter_markers": chapter_marker_count,
    "paragraph_lead_lines": paragraph_lead_count,
    "blank_lines": blank_line_count,
    "raw_paragraphs": len(raw_paragraphs),
    "processed_segments": len(processed_paragraphs),
}

source_diagnostics

## Standard segment dataframe

The segmentation step is standardised into a dataframe named `segments_df`. The later sentiment-scoring cells use this dataframe rather than depending on the specific preprocessing method that produced the segments.

At minimum, `segments_df` contains the segment text and aligned chapter label. A `segment_type` column is included so that future versions can distinguish narration, dialogue, verse, or mixed segments.

In the current plain-text parser, final merged segments are labelled as `mixed_or_unknown`. Future versions may propagate `narration`, `dialogue`, or `verse` labels into the final dataframe.

In [ ]:
# Cell — Build standard segments dataframe

segments_df = pd.DataFrame(
    {
        "segment_text": processed_paragraphs,
        "chapter": chapter_tags,
        "segment_type": "mixed_or_unknown",
    }
)

segments_df["token_count"] = segments_df["segment_text"].apply(tokenize_count)

segments_df.head()

In [ ]:
# Check segment counts by chapter
chapter_segment_counts = (
    segments_df["chapter"]
    .value_counts()
    .sort_index()
)

chapter_segment_counts.head()

In [ ]:
# Cell — Segmentation diagnostics

segment_lengths = [tokenize_count(segment) for segment in processed_paragraphs]

diagnostics = {
    "raw_paragraphs": len(raw_paragraphs),
    "processed_segments": len(processed_paragraphs),
    "unique_chapters": len(set(chapter_tags)),
    "min_segment_tokens": min(segment_lengths),
    "max_segment_tokens": max(segment_lengths),
    "mean_segment_tokens": np.mean(segment_lengths),
    "median_segment_tokens": np.median(segment_lengths),
}

diagnostics

## Anchored sentiment scoring

This section scores each segment by comparing it with positive and negative anchor sentences.

For each segment, the model computes:

`sentiment_score = mean_similarity_to_positive_anchors - mean_similarity_to_negative_anchors`

Positive scores indicate that a segment is closer to the positive anchors, while negative scores indicate that it is closer to the negative anchors.

The anchor sentences themselves are not included in the public repository because they are drawn from copyrighted text.

In [ ]:
# Cell — Anchored sentiment scoring function

def compute_anchored_sentiment_scores(
    model: SentenceTransformer,
    segments: list[str],
    positive_anchors: list[str],
    negative_anchors: list[str],
    batch_size: int = 64,
) -> np.ndarray:
    """
    Compute anchored sentiment scores for text segments.

    Score = mean cosine similarity to positive anchors
            minus mean cosine similarity to negative anchors.
    """
    if not positive_anchors:
        raise ValueError("positive_anchors is empty.")

    if not negative_anchors:
        raise ValueError("negative_anchors is empty.")

    positive_anchor_vectors = model.encode(
        positive_anchors,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    negative_anchor_vectors = model.encode(
        negative_anchors,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    segment_vectors = model.encode(
        segments,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    positive_similarity = util.cos_sim(
        segment_vectors,
        positive_anchor_vectors,
    ).mean(dim=1)

    negative_similarity = util.cos_sim(
        segment_vectors,
        negative_anchor_vectors,
    ).mean(dim=1)

    scores = positive_similarity - negative_similarity

    return scores.cpu().numpy()

In [ ]:
# Cell — Score corpus segments

MODEL_PATH = "../models/tolkien_sentence_transformer_epoch_1"  # Edit this path if needed.
model = SentenceTransformer(MODEL_PATH)

ANCHORS_PATH = Path("data/anchors.csv")

if not ANCHORS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {ANCHORS_PATH}. "
        "Anchor sentences are not included in the public repository because they contain copyrighted text."
    )

anchors_df = pd.read_csv(ANCHORS_PATH, sep=";")

positive_anchors = anchors_df.loc[
    anchors_df["label"] == "positive",
    "sentence_text",
].tolist()

negative_anchors = anchors_df.loc[
    anchors_df["label"] == "negative",
    "sentence_text",
].tolist()

segments_df["sentiment_score"] = compute_anchored_sentiment_scores(
    model=model,
    segments=segments_df["segment_text"].tolist(),
    positive_anchors=positive_anchors,
    negative_anchors=negative_anchors,
    batch_size=64,
)

segments_df.head()

In [ ]:
# Cell — Inspect strongest positive and negative segments

display_cols = ["chapter", "segment_type", "token_count", "sentiment_score", "segment_text"]

most_positive = (
    segments_df
    .sort_values("sentiment_score", ascending=False)
    [display_cols]
    .head(10)
)

most_negative = (
    segments_df
    .sort_values("sentiment_score", ascending=True)
    [display_cols]
    .head(10)
)

most_positive, most_negative

## Visualising the sentiment trajectory

The sentiment trajectory is plotted over the ordered sequence of analysis segments. A rolling mean is used to smooth local variation and make broader narrative movement easier to inspect.

The smoothing window is a visualisation parameter rather than part of the model itself. It should be reported whenever figures generated from this notebook are used in analysis.

In [ ]:
# Cell — Prepare smoothed trajectory

WINDOW = 20

plot_df = segments_df.copy()
plot_df["segment_index"] = range(len(plot_df))
plot_df["smoothed_score"] = (
    plot_df["sentiment_score"]
    .rolling(window=WINDOW, center=True, min_periods=1)
    .mean()
)

plot_df.head()

In [ ]:
# Cell — Trajectory plotting function

import matplotlib.pyplot as plt
import numpy as np

def plot_sentiment_trajectory(
    df: pd.DataFrame,
    score_col: str = "smoothed_score",
    chapter_col: str = "chapter",
    title: str = "Anchored sentiment trajectory",
    figsize: tuple[int, int] = (18, 6),
):
    """
    Plot an anchored sentiment trajectory over ordered text segments.

    Positive regions are shaded above zero and negative regions below zero.
    Chapter labels are shown at chapter midpoints.
    """
    required_cols = {"segment_index", score_col, chapter_col}
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns for plotting: {missing}")

    x = df["segment_index"].to_numpy()
    y = df[score_col].to_numpy()

    fig, ax = plt.subplots(figsize=figsize)

    ax.plot(
    x,
    y,
    color="black",
    linewidth=1.8,
    label=f"{score_col} (window={WINDOW})",
)
    ax.axhline(0, linewidth=1, linestyle="--")

    ax.fill_between(
    x,
    y,
    0,
    where=y >= 0,
    color="steelblue",
    alpha=0.25,
    interpolate=True,
    label="positive region",
)

    ax.fill_between(
    x,
    y,
    0,
    where=y < 0,
    color="indianred",
    alpha=0.25,
    interpolate=True,
    label="negative region",
)

    # Chapter boundaries and labels
    chapter_starts = df.groupby(chapter_col)["segment_index"].min()
    chapter_mids = df.groupby(chapter_col)["segment_index"].median()

    for chapter, start in chapter_starts.items():
        ax.axvline(start, linewidth=0.5, alpha=0.25)

    ax.set_xticks(chapter_mids.values)
    ax.set_xticklabels(chapter_mids.index, rotation=90, fontsize=8)

    ax.set_xlabel("Chapter")
    ax.set_ylabel("Anchored sentiment score")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell — Plot full trajectory

plot_sentiment_trajectory(
    plot_df,
    score_col="smoothed_score",
    chapter_col="chapter",
    title=f"Anchored sentiment trajectory (rolling window = {WINDOW})",
)